# SQL

SQL (Standard Query Language) also known as RDBMS (Relational Database Management System) are databases that store persistent data using tables. This provides a way to structure data within the disk so that reading from and writing to is efficient.

## B+ Trees
What makes these queries efficient is the user of a B+ Tree. These trees are multi-way trees that can have more thant two children per node, and store all their data in the leaf nodes which are linked in a sorted order. All of the data is stored in the leaf nodes. 

![alt text](image/b_tree.png)

A reason why B+ trees are used is because they provide indexing which is a way to improve the speed of data retrieval operations on database table. It comes at the cost of additional writes and storage space to maintain the index data strcture.

### How data is stored

In a SQL database, data is stored inside tables. Tables are a way to organise data where each row contains information about a single primary key. A primary key uniquely identifies each record, where a record is a row.

```SQL
# creating a table
CREATE TABLE People (
    PhoneNumber int PRIMARY KEY,
    Name varchar(100)
);
```

```SQL
# creating a table that refers to the Homes table to retrieve the Phone Number
CREATE TABLE Homes (
    PhoneNumber int,
    Address varchar(255),
    FOREIGN KEY (PhoneNumber) REFERENCES People(PhoneNumber)
);
```
Entity Relationship Diagram (ERD)

<img src="image/erd.png" width="800">

### Joins
If we wanted to retrieve the names and addresses of the people based on their phone numbers, we would need to perform a join operation

```SQL
SELECT People.Name, Homes.Address
FROM People
JOIN Homes ON People.PhoneNumber = Homes.PhoneNumber;
``


## Trade-offs of RDBMS

RDBMS follows ACID which is a property that stands for Atomicity, Consistency, Isolation and Durability. These refers to key properties of a transaction which refers to a sequence of one or more SQL opereations.



- Atmocity: All changes to data are performed as if they are a single operation. When we have the `Begin` and `Commit`. If one of the steps fail in the transaction, it means that the entire transaction is not commited. It is either all of the steps get done, or none of the steps gets done.

- Consistency: This ensures data integrity. The database follows predefined rules and constraints that maintain the vailidity of the data throughout the execution of multiple transactions. This can be done by defining what constraints should be followed such as which account balances cannot be negative or have a non-null name

- Isolation: This refers to an intermediate state, such that a transaction is invisible to other transactions. This property ensures that the concurrent transactions do not interfere with each other and the effects of one transaction are isolated from other transactions until it is committed.

Suppose Alice has $1000 in her acocount and Bob has $500. 

First transaction: 
1. Deduct $500 from Alice
2. The addition of the withdrawn $500 from Alice's account to Bob's account.
3. Commit

However, before this happens there is a second transaction which adds $220 to Alice's account and commits. This now violates the isolate property which result in a condition called dirty read. Which occurs when one transaction reads data from another transaction before it has been commited. This now means that transaction 2 reads a value from a transaction that was not yet committed.

To maintain isolation, the second transaction should have waited until the first transaction committed before reading and committting any changes to it.

- Durability: After a transcation succesfully completes, changes to data persist are not undone, even in the event of a system failure. This would mean that if in an application that trasnfers funs from one account to another, all transactions will be remembered. Even if the power goes out after a transaction, the transaction will still be recorded in the database.



# NoSQL

noSQL (Not-Only SQL) and also known as non-relational databases are much more flexible and scalable thatn SQL databases and were designed to overcome the limitations of SQL databases such as scalabilitiy and performance. NoSQL databases can be scaled horizontaly, which is a signifciant advantage when managing large applications.

## Key-Value Databases
This is a database that uses a simple key-value method to store data. Just like a hashmap, the database stores a collection of key-value pairs and the key serves as a unique identifier. Both the key and the values can range from simgple to complext objects. Key-Value databases are schemaless (Meaning it manages information without the need for a blueprint) and different keys and values can have completely different structures. One key might be a string and map to a JSON object and canother can be an integer and map to a list.

```python
Key: 'user:123'
Value: name=John Doe, age=30, email=johndoe@example.com

Key: 'product:456'
Value: name=Widget, price=9.99, description=A useful widget for various purposes.

Key: 'order:789'
Value: customer_id=user:123, products=product:456|product:789, total_amount=29.98
```

A good example is redis, where it stores data in RAM. This makes it exceptionally fast for data retrieval since RAM is significalntly faster compared to disk-based storage.

An example would be

```python
# the database being the dictionary
data = {
    "user:1": {...}
}
```

```python
# and the lookup is just searching using the key which is an O(1) operation
data['user:1']
```

## Document Databases

These databases store data as 'documents'. These 'documents' are JSON-like making it easier for developers to store and query data because the format that the database stores data is in the same document-model format developers use in their application code.

An example below is:
```python
{
  "field1": {
    "onetype" : [
      {"id": 1, "name":"John Doe"},
      {"id": 2, "name":"Don Joeh"},
    ],
    "othertype": {"id": 2, "company": "ABC"}
    },
    "field2": {
    "list1": [[1,42],[2,2]]
  }
}
```

A Dcoument Database example is MongoDB

## Wide-Column Databases

These stores data in columns rather than in rows. Each column corresponds to a specific attribute or field of the data. The structure enables high write throughput and optimizes read and aggregations over a subset of data. If an application needs to manage alot of timestamp data, wide column databases excels in handling that.

```python
# Database
id | name | age
----------------
1  | Joey | 25
2  | Alex | 30
```

```
# Storing in Row-based
(1, Joey, 25), (2, Alex, 30)
```

```
# Storing in Column-based
id   → [1, 2]
name → [Joey, Alex]
age  → [25, 30]
```

An example is Apach Cassandra


## Graph databases

These databases uses a graph like structure where each node refers to an entity. Much like graphs in algo, nodes in a graph database are connected to each other through edges or relationships. They can be used to represent various types of relationship, friendships and assosications between entities. Graph databases are very much useful when the data has complex relationship and interconnectedness. 

<img src="image/graphdb.png" width="800">


## Why is there a need for NoSQL databases

Because there are no foregin keys or join constraints, the data can be split and stored on different server. This means half the data can be stored in one database, the other other half can be store in another part of the world. This database is designed with distributed architecture in mind. 

ACID focuses on strict consistency within the databases, BASE focuses more on eventual consistency. That is not to say one is better than the other but ACID is more common in SQL databases and BASE describes NoSQL better. BASE stands for "Basically Available, Soft state and Eventual Consistency"

## Eventual Consistency

A mechanism that provides this is the leader/follower archiecture. Updates and writes can only be done to the primary node and the primary node eventually updates the rest of the nodes. In this case, the primary node can be said to be the `leader` and the rest of the nodes being the `followers`. Because the nodes, aprat from the primary node are only eventually being updated, there might be times when a user requests data and it is stale. This can be true in apps like Twitter or Instagram when updating the follower count may be delayed because the leader node has not updated the rest of the nodes.

<img src="image/ec.png" width="800">
<img src="image/sc.png" width="800">


## Replicaton

Replication creates a copy of the database called a replica. These replica(s) are hosted on a seperate machine or server and it is kepy in sync with the original database. This method is used to increase data availability, improve scalability, and increase data durability. 

The original database is the leader/master while the replice is the follower/slave.

In this architecture, data replication flows from the leader to the follower and the leader is responsible for updating the follwoer. If replication were attempted from the follower to the leader, the leader will not be fully updated

## Synchronous Replication

In sync replication, every write transaction to the leader is immediately replicated to the follower, ensuring consistency betweem two replicas. But this introduce latency, as the write process will not be completed until the leader has completely sync with the rest of the follower on the replication. THe benefit is that if the leader goes down, the mostly updated follower can take its place.

<img src="image/sync.png" width="700">


## Asynchronous Replication

This replication invovles a delay in data replication. The leader database commits the transation and sends the replication data tot he follower without waiting for the follower to acknowledge or apply the changes immediately. This reduces latency, but it menas if a client makes a request to the follower before it has been updated, the data might not be updated. This is the trade-off made for increased availability.


<img src="image/async.png" width="700">



# Master-Master (Multi-Master) Replication

This is used when the data needs to be served in different regions. For example one might serve the west and the other sever the east. Both leaders can be written and read from, making it ideal for distributing data across different parts of the world. However, synchronization latency between the leaders can be a challenge, and measures like periodic updates are needed to keep them in sync.

<img src="image/mm.png" width="700">


## Sharding

Sharding is used when replication alone is insufficient to handle the high traffic volume on a single database. It invovles dividng the database into smaller shards where each is hosted on a serpeate machine or server. By distributing the data and workload across different shards, the system achieves improved peroframnce, scalability and availability. Each shard contains only a subset of the entire dataset, and they do not have a complete copy of the original database.


## Different approaches to sharding

## Range-Based Shard

Data is split according to ranges, if we apply this to our 100 row approach with our four shards, we might split it by having 1-25, 26-50, 51-75, 76-100, where these numbers are the IDs. Determining how data is paritioned among shards is done using a shard key. A shard key is a chosen attribute that determines which shard each data belong to. Another example is that the shard key might based on sex, splittin the data between male and female. Or it could based on first name/ last name basis.


<img src="image/rbs.png" width="700">

## Challenges with sharding

WHen dealing with hundreds of tables, ensuring related data end up in the same shard can be complex. Additionally, maintaining the ACID properties poses challenges in sharing relational databases. Therefore SQL database like MySQL and PostgresSQL do not inherenetly support sharding, requiring devlopers to implement sharding logic at the applcation level which can be complicated.

However, NoSQL databases are designed with horizontal scaling in mind and are better suited for sharding. 

## CAP Theorem

Consistency, Availability and Partition Tolerance. This concept suggests that a distributed system containing a `Master` and `Slave` can only ensure two out of these three guarantees simultaneously, either Parition Tolernace and Consistency or Partition Tolerance and Availability, but never all three at once.

### Partiion Tolerance

A partition in a distributed system arises when a communication breakdown between the `leader` and the `follower` nodes prevenets the `leader` from updating the follower. Various factors can trigger this such as network failure, hardware issue. If a system demostrates partition tolerance, it implies that it can persist in functioning despite network failures, thereby avoiding a total system collaps.

### Conssitency

Consistency refers to the unfiromity of data between the leader and follower (unlike in ACID where it refers to the data being consistent in terms of it being unable to be negative or NULL in SQL). This ensures that all nodes within the system percieve the data identically at any given moment

### Availability

Availability refers to that every system request recieves a request, be it successful or a failure. It ensures that the system stays operational and can manage requests even amid failures.


### Availability over Consistency
<img src="image/aoc.png" width="700">


### Consistency over Availability 
<img src="image/coa.png" width="700">


## PACELC Theorem (An extension of the CAP Theroem)

"Given P (a network partiion), choose A (availability) or C (consistency). Else favor Latency or Consistency"

Latency because the `leader` is updating the data in other `followers`.

<img src="image/pacelc.png" width="500">


# Object Storage

Object storage treats each piece of data as an object, comprising the actual data, metadata, and a unique identifier. There is no strict hierarchy in object storage compared to a file system, as objects are stored in a flat address space. This facilitates easier scalability compared to file storage systems. Object storage was developed to address the limitations of storing BLOB (Binary Large Object) data in databases and is commonly used for storing items such as images, videos, and database backups.

From a design perspective, storing images or videos in a database is generally not recommended as it can hinder performance, increase storage requirements, and increase I/O overhead on the database. Traditional RDBMS are not optimized for handling large files, whereas object storage systems are designed to handle unstructured data efficiently. One significant advantage of using object-based storage is its scalability, allowing for easy scaling of the flat architecture without encountering the limitations associated with file storage.

When retrieving data from an object store, it is typically done through HTTP-based APIs or SDKs.